# HyperKT Stage 3B — Hypergraph Collapse Diagnosis and Repair
**Runtime:** Google Colab **T4 GPU** (CPU fallback). Upload `hyperkt_stage2.zip` and `hyperkt_stage3.zip`. This notebook evaluates a frozen training-only co-occurrence graph; held-out edges are a structural validation diagnostic, **not** independent student-level generalization. Never report these scores as knowledge tracing AUC.

In [ ]:
import os, io, json, zipfile, random, hashlib, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy.sparse as sp
import torch
from torch import nn
from google.colab import files
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Upload BOTH stage2.zip and stage3.zip')
uploaded=files.upload()
assert 'hyperkt_stage2.zip' in uploaded and 'hyperkt_stage3.zip' in uploaded
z2=zipfile.ZipFile(io.BytesIO(uploaded['hyperkt_stage2.zip']))
z3=zipfile.ZipFile(io.BytesIO(uploaded['hyperkt_stage3.zip']))
for z in (z2,z3): assert z.testzip() is None
old=np.load(io.BytesIO(z3.read('stage3_hypergraph.npz')))
obs=old['observed'].astype(np.float64)
assert obs.shape[0]==obs.shape[1]==len(old['skill_ids'])
assert np.isfinite(obs).all()
print('Graph shape:',obs.shape,'min/max:',obs.min(),obs.max())

## 1. Audit the original model against trivial baselines

In [ ]:
N=len(obs); tri=np.triu_indices(N,1)
y=obs[tri]; pred=old['reconstruction'][tri].astype(float)
constant=np.full_like(y,y.mean()); zero=np.zeros_like(y)
def mse(a,b):return float(np.mean((a-b)**2))
H=old['assignments'].astype(float)
P=np.clip(H,1e-12,None);P=P/P.sum(axis=1,keepdims=True)
entropy=-np.sum(P*np.log(P),axis=1)/np.log(P.shape[1])
report_old={'old_mse':mse(y,pred),'constant_mse':mse(y,constant),'zero_mse':mse(y,zero),
'old_prediction_std':float(pred.std()),'observed_std':float(y.std()),
'assignment_normalized_entropy_mean':float(entropy.mean()),
'assignment_row_std_mean':float(P.std(axis=1).mean()),
'positive_pair_fraction':float((y>0).mean())}
print(json.dumps(report_old,indent=2))
fig,ax=plt.subplots(1,2,figsize=(10,4));ax[0].hist(y,bins=40);ax[0].set_title('Observed pair weights');ax[1].scatter(y,pred,s=2,alpha=.2);ax[1].set_title('Old reconstruction vs observed');ax[1].set_xlabel('Observed');ax[1].set_ylabel('Predicted');plt.tight_layout();plt.show()

## 2. Define held-out positive and negative skill pairs
The observed matrix is derived entirely from **training interactions**. We hold out graph entries only to detect structural overfitting. Negative means **unobserved co-occurrence**, not a confirmed absence of a relationship. Do not use validation/test student interactions to construct this graph.

In [ ]:
# Binary adjacency: any observed off-diagonal co-occurrence counts as an observed pair.
A=(obs>0).astype(np.float32);np.fill_diagonal(A,0)
positive=np.argwhere(np.triu(A,1)>0)
negative=np.argwhere(np.triu(1-A,1)>0)
rng=np.random.default_rng(SEED)
rng.shuffle(positive);rng.shuffle(negative)
def split_pairs(arr):
    n=len(arr);n_val=max(1,round(.15*n));n_test=max(1,round(.15*n))
    return arr[n_val+n_test:],arr[:n_val],arr[n_val:n_val+n_test]
p_tr,p_val,p_test=split_pairs(positive);n_tr,n_val,n_test=split_pairs(negative)
assert min(map(len,[p_tr,p_val,p_test,n_tr,n_val,n_test]))>0
print('Positive train/val/test:',list(map(len,[p_tr,p_val,p_test])))
print('Unobserved train/val/test:',list(map(len,[n_tr,n_val,n_test])))
# Graph used to initialize skill embeddings must NOT include held-out positive pairs.
A_train=A.copy()
for pairs in [p_val,p_test]:
    A_train[pairs[:,0],pairs[:,1]]=0;A_train[pairs[:,1],pairs[:,0]]=0
assert np.all(A_train[p_val[:,0],p_val[:,1]]==0)
assert np.all(A_train[p_test[:,0],p_test[:,1]]==0)

## 3. Learn a nonnegative, low-rank hyperedge affinity with class-balanced pair loss
Each skill has a nonnegative hyperedge membership. Pair logits use a normalized dot product and trainable global scale/bias. Sparsity and diversity penalties discourage identical assignments. The regularizers do not guarantee meaningful pedagogical relationships; validation and ablations remain necessary.

In [ ]:
class Hypergraph(nn.Module):
    def __init__(self,n,k=32):
        super().__init__();self.logits=nn.Parameter(torch.randn(n,k)*.15)
        self.scale=nn.Parameter(torch.tensor(2.0));self.bias=nn.Parameter(torch.tensor(-1.0))
    def forward(self,pairs):
        h=torch.softmax(self.logits,dim=-1)
        similarity=(h[pairs[:,0]]*h[pairs[:,1]]).sum(-1)*h.shape[1]
        return self.scale.clamp(.01,20)*similarity+self.bias,h
model=Hypergraph(N,32).to(device)
def pairs_tensor(pos,neg):
    pairs=np.concatenate([pos,neg]);labels=np.r_[np.ones(len(pos)),np.zeros(len(neg))].astype(np.float32)
    return torch.as_tensor(pairs,device=device,dtype=torch.long),torch.as_tensor(labels,device=device)
train_pairs,train_labels=pairs_tensor(p_tr,n_tr)
val_pairs,val_labels=pairs_tensor(p_val,n_val)
test_pairs,test_labels=pairs_tensor(p_test,n_test)
from sklearn.metrics import roc_auc_score,average_precision_score,log_loss
@torch.no_grad()
def evaluate(pairs,labels):
    model.eval();logits,h=model(pairs);prob=torch.sigmoid(logits).cpu().numpy();target=labels.cpu().numpy()
    return {'auc':float(roc_auc_score(target,prob)), 'ap':float(average_precision_score(target,prob)),
            'bce':float(log_loss(target,np.clip(prob,1e-7,1-1e-7))), 'pred_std':float(prob.std())}
opt=torch.optim.AdamW(model.parameters(),lr=.01,weight_decay=.001)
# All pair labels retained; positive class weighted for sparse graph.
weight=torch.tensor(len(n_tr)/max(1,len(p_tr)),device=device,dtype=torch.float32)
loss_fn=nn.BCEWithLogitsLoss(pos_weight=weight)
best=float('inf');best_state=None;wait=0;patience=60;history=[]
for epoch in range(1,801):
    model.train();opt.zero_grad();logits,h=model(train_pairs)
    reconstruction=loss_fn(logits,train_labels)
    # Promote assignment diversity without imposing fabricated prerequisite directions.
    diversity=((h.mean(0)-1/h.shape[1])**2).mean()
    concentration=(h*(1-h)).sum(-1).mean()
    loss=reconstruction+0.05*concentration+0.1*diversity
    loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
    v=evaluate(val_pairs,val_labels)
    history.append({'epoch':epoch,'train_loss':float(loss.item()),'val_bce':v['bce'],'val_auc':v['auc']})
    if v['bce']<best-1e-5:
        best=v['bce'];best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};best_epoch=epoch;wait=0
    else:wait+=1
    if epoch%50==0:print(epoch,'loss',round(float(loss.item()),4),'val_auc',round(v['auc'],4))
    if wait>=patience:break
model.load_state_dict(best_state)
print('Best epoch',best_epoch,'Validation',evaluate(val_pairs,val_labels))

## 4. Frozen evaluation, baselines, collapse diagnostics

In [ ]:
from sklearn.metrics import roc_auc_score,average_precision_score
# Evaluate once after model selection; do not tune using the test split.
val=evaluate(val_pairs,val_labels);test=evaluate(test_pairs,test_labels)
with torch.no_grad():
    h=torch.softmax(model.logits,dim=-1).cpu().numpy()
    logits_all,_=model(torch.as_tensor(np.column_stack(tri),device=device,dtype=torch.long))
    all_scores=torch.sigmoid(logits_all).cpu().numpy()
row_entropy=float((-h*np.log(np.maximum(h,1e-12))).sum(1).mean()/np.log(h.shape[1]))
# Degree-product heuristic trained only on visible graph edges.
deg=A_train.sum(1)
def baseline(pairs,labels):
    score=deg[pairs[:,0]]*deg[pairs[:,1]]
    return {'auc':float(roc_auc_score(labels.cpu().numpy(),score)),
            'ap':float(average_precision_score(labels.cpu().numpy(),score))}
results={'old_diagnostics':report_old,'new_validation':val,'new_test':test,
'degree_product_test_baseline':baseline(test_pairs,test_labels),
'best_epoch':best_epoch,'membership_entropy_normalized':row_entropy,
'new_prediction_std_all_pairs':float(all_scores.std()),
'positive_train':len(p_tr),'positive_val':len(p_val),'positive_test':len(p_test),
'negative_train':len(n_tr),'negative_val':len(n_val),'negative_test':len(n_test)}
print(json.dumps(results,indent=2))
if test['auc']<=results['degree_product_test_baseline']['auc']:
 print('WARNING: Hypergraph does not outperform degree-product baseline on held-out edges.')
if row_entropy>.95 or all_scores.std()<.02:
 print('WARNING: Memberships or scores may remain nearly uniform.')
fig,ax=plt.subplots(1,2,figsize=(11,4));hist=pd.DataFrame(history)
ax[0].plot(hist.epoch,hist.train_loss,label='Train objective');ax[0].plot(hist.epoch,hist.val_bce,label='Validation BCE');ax[0].legend();ax[0].set_xlabel('Epoch')
ax[1].imshow(h,aspect='auto',interpolation='nearest');ax[1].set_xlabel('Hyperedge');ax[1].set_ylabel('Skill');plt.tight_layout();plt.show()

## 5. Export reproducible Stage 3B artifacts
The model is an **undirected structural affinity model**, not proof of directed skill prerequisites. If Stage 3B does not outperform simple baselines, we should not claim successful hypergraph discovery.

In [ ]:
out='/content/hyperkt_stage3b';os.makedirs(out,exist_ok=True)
np.savez_compressed(f'{out}/hypergraph_repaired.npz',skill_ids=old['skill_ids'],assignments=h.astype(np.float32),
 observed=obs.astype(np.float32),train_visible_adjacency=A_train,
 validation_positive=p_val,validation_unobserved=n_val,test_positive=p_test,test_unobserved=n_test)
torch.save({'state_dict':best_state,'n_skills':N,'n_edges':32,'seed':SEED,'best_epoch':best_epoch},f'{out}/hypergraph_repaired.pt')
pd.DataFrame(history).to_csv(f'{out}/training_history.csv',index=False)
with open(f'{out}/report.json','w') as f:json.dump(results,f,indent=2)
with open(f'{out}/protocol.txt','w') as f:f.write('Training-interaction-only co-occurrence; held-out pair diagnostics, not independent student generalization. Undirected associations, not causal prerequisites. Random seed 42.\n')
shutil.make_archive('/content/hyperkt_stage3b','zip',out)
print('Created /content/hyperkt_stage3b.zip')
files.download('/content/hyperkt_stage3b.zip')